# Water demand comparison + WSI historical


In [ ]:
# basic
from pathlib import Path

import pandas as pd

# spatial
import geopandas as gpd
import xarray as xr

# plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# metrics
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(7, 25))
from src.utils import load_config, weighted_mean_by_lat, extract_attributes


cfg = load_config()
path_data_raw = Path(cfg["dir"]["data"])

In [ ]:
scenarios       = [s["id"] for s in cfg["scenarios"]]
scenarios_names = [s["name"] for s in cfg["scenarios"]]
colors          = [s["color"] for s in cfg["scenarios"]]

## Data

In [ ]:
# metadata
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")#.set_index("gauge_id")
hydroatlas_shape["basin_cenlat_rounded"] = hydroatlas_shape["basin_cenlat"].round(0)
common_period = slice("1990-01-01", "2010-12-31")

# local water demand data
ds_cr2wu_nlu    = xr.open_dataset(path_data_raw / "water_use/CR2WU/CR2WU_NLU_1960_2020.nc").Total_NLU
ds_cr2wu_lulucf = xr.open_dataset(path_data_raw / "water_use/CR2WU/CR2WU_LULUCF_1960_2020.nc").cons_total
ds_cr2wu = ds_cr2wu_nlu + ds_cr2wu_lulucf
ds_cr2wu = ds_cr2wu.sel(time = common_period)
ds_cr2wu = ds_cr2wu.mean(dim = "time")

# global water demand data (uncorrected)
ds_global_irr  = xr.open_dataset(path_data_raw / "water_use/Huang_HESS/Water_use_Huang.nc").cons_irr
ds_global_irr  = ds_global_irr.sel(date=common_period) # overlappping period with runoff data
ds_global_irr  = ds_global_irr.resample(date="YS").sum() # annual sum
ds_global_irr  = ds_global_irr.mean(dim = "date")

ds_global  = xr.open_dataset(path_data_raw / "water_use/Huang_HESS/Water_use_Huang.nc").cons_total
ds_global  = ds_global.sel(date=common_period) # overlappping period with runoff data
ds_global  = ds_global.resample(date="YS").sum() # annual sum
ds_global  = ds_global.mean(dim = "date")
ds_global  = ds_global.interp(lon = ds_cr2wu.x, lat = ds_cr2wu.y, method="nearest") # for better comparison
ds_global  = ds_global.where(ds_cr2wu >= 0)

In [ ]:
## Comparison at the basin scale
hydroatlas_shape = extract_attributes(ds_cr2wu, hydroatlas_shape, "ds_cr2wu")
hydroatlas_shape = extract_attributes(ds_global, hydroatlas_shape, "ds_global")
hydroatlas_shape = extract_attributes(ds_global_irr, hydroatlas_shape, "ds_global_irr")

# Performance metrics
valid = hydroatlas_shape.dropna(subset=["ds_cr2wu", "ds_global"])
r_pearson, _ = pearsonr(valid.ds_cr2wu, valid.ds_global)
mae   = mean_absolute_error(valid.ds_cr2wu, valid.ds_global)
pbias = 100 * (valid.ds_global.sum() - valid.ds_cr2wu.sum()) \
              / valid.ds_cr2wu.sum()

print(f"Pearson-r: {r_pearson:.2f}")
print(f"MAE: {mae:.2f}")
print(f"Percent Bias: {pbias:.2f}%")

In [ ]:
# water stress data (this is done with the corrected water demand data)
wsi = pd.read_parquet("data/hydro/water_stress.parquet")
wsi = wsi.set_index(["scenario", "month", "gauge_id"])[["WSI"]]
#wsi = wsi.groupby(["scenario", "gauge_id"]).max()
#wsi = wsi.loc[wsi.index.get_level_values("month").isin([1, 2, 12])]
wsi = wsi.loc[wsi.index.get_level_values("scenario").isin(["historical"])]
wsi = wsi.groupby(["gauge_id"]).mean()

hydroatlas_shape = hydroatlas_shape.set_index("gauge_id").join(wsi)
hydroatlas_shape_lat = weighted_mean_by_lat(hydroatlas_shape)

## Plot

In [ ]:
fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.1, vertical_spacing=0.09,
                    column_widths=[0.5, 0.5],
                    specs=[[{"type": "xy", "rowspan": 2}, {"type": "xy"}],
                           [None, {"type": "xy"}]])

# Panel 1: Water Stress Index (WSI)
fig.add_trace(go.Bar(x=hydroatlas_shape_lat["WSI"]*100, y=hydroatlas_shape_lat.index, 
                        marker_color="black", opacity=0.6, orientation='h', showlegend=False), 
                row=1, col=1)

fig.update_yaxes(range = [-56, -7], ticksuffix = "ºS", row =1, col=1)
fig.update_xaxes(title_text="Water Stress Index (WSI)", dtick = 25, range = [0, 100], title_standoff = 0, row=1, col=1)

color = "#474747"
fig.add_vline(x=40, line_width=1, line_dash="dot", line_color=color, row=1, col=1)
fig.add_annotation(x=32, y=-7.5, xref="x", yref="y", text="High stress", showarrow=False, font_family="Arial", 
                   font=dict(size=14, color=color), xanchor="left", yanchor="top", textangle=-90)
fig.add_vline(x=70, line_width=1, line_dash="dot", line_color=color, row=1, col=1)
fig.add_annotation(x=62, y=-7.5, xref="x", yref="y", text="Extreme stress", showarrow=False, font_family="Arial", 
                   font=dict(size=14, color=color), xanchor="left", yanchor="top", textangle=-90)
fig.add_vline(x=100, line_width=1, line_dash="dot", line_color=color, row=1, col=1)
fig.add_annotation(x=92, y=-7.5, xref="x", yref="y", text="Overuse", showarrow=False, font_family="Arial", 
                   font=dict(size=14, color=color), xanchor="left", yanchor="top", textangle=-90)

# Panel 2: Scatter plot
fig.add_trace(go.Scatter(x=[-30, 500], y=[-30, 500],
                         mode="lines", name="1:1 line",
                         line=dict(color="#BAC7DA", dash='dot'),
                         showlegend=False), row=1, col=2)


fig.add_trace(go.Scatter(x=valid["ds_global"], y=valid["ds_cr2wu"],
                         mode="markers", name="Basins",
                         marker=dict(color=valid['ds_global_irr'], 
                                     colorscale="aggrnyl_r",
                                     line = dict(color="#40663C", width=0.2), cmax = 250, cmin = 0,
                                     colorbar=dict(title="Irrigation<br>(mm yr<sup>-1</sup>)<br>", thickness=18, tickwidth=0.2),
                                     size=scaler.fit_transform(valid[['basin_area']]).flatten(),
                                     opacity=1), showlegend=False), row=1, col=2)


fig.update_xaxes(title_text="Huang et al. (mm yr<sup>-1</sup>)", range = [-30, 500], dtick = 100,  row=1, col=2)
fig.update_yaxes(title_text="CR2WU (mm yr<sup>-1</sup>)", range = [-30, 500], dtick = 100, row=1, col=2)


fig.add_annotation(text=f"r-Pearson = {r_pearson:.2f}<br>MAE = {mae:.2f}<br>PBIAS = {pbias:.2f}%",
                   xref="x", yref="y", x=400, y=20, bgcolor="rgba(255, 255, 255, 0.8)",
                   showarrow=False, align="left",
                   font=dict(size=13), row=1, col=2)


# Panel 3: Histogram of difference
fig.add_trace(go.Histogram(x=valid["ds_cr2wu"] - valid["ds_global"], 
                           histnorm='percent',
                           nbinsx=50, marker_color="#326D4C", 
                           opacity=0.9, showlegend=False), row=2, col=2)

fig.update_xaxes(title_text="Difference (mm yr<sup>-1</sup>)", range = [-150, 150], row=2, col=2)
fig.update_yaxes(title_text="Percent (%)", row=2, col=2)



# Add annotations for each panel
annotations = [
    ("a)", 0.01, 0.99),
    ("b)", 0.56, 0.99),
    ("c)", 0.56, 0.45),
]

for text, x, y in annotations:
    fig.add_annotation(text=text, x=x, y=y, xref="paper", yref="paper",
                       showarrow=False, font=dict(size=15, family="Arial", color="black"),
                       xanchor="left", yanchor="top")

# Update settings for each subplot
fig.update_xaxes(ticks="outside", griddash="dot", title_standoff = 1,  zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", title_standoff = 1, zeroline=False)


fig.update_layout(width=800, height=700, margin=dict(l=10, r=10, b=10, t=10), 
                  barmode='overlay', font_family="Arial", 
                      plot_bgcolor="rgba(229,236,246,0.7)", font_size=14)

fig.write_image("figures/fig_water_stress.svg", width=800, height=700)
fig.show()